# best practices

In [ ]:
import re
import pandas as pd

input_data_path = '../../test_data/test_data.csv'
data_output_dir = '../../test_data/25_best_practices/'


def fix_column_names(name):
    name = re.sub(r'[\s\-/]', '_', name) # Replace whitespace & - characters with '_'d
    name = re.sub(r'[^\w\s]', '', name) # Remove all non-alphanumeric characters except '_'
    return name.lower()

# good READ example
df = (
    pd.read_csv(
        input_data_path, 
        index_col=['ID'], 
        parse_dates=['Birthday']
    )
    .rename(columns=fix_column_names)
    # .rename(columns={'Name': 'name', 'Age': 'age', ... })
)

# good WRITE example 1
df.to_csv(data_output_dir + 'good_example_1.csv', index=False)
"""
it's best to store large files as something other than csv

large_df.to_parquet('path/to/data.parquet')
large_df.to_feather('path/to/data.feather')
large_df.to_pickle('path/to/data.pickle')
"""

print(df.info())
df.head()

## Don't use spaces in column names
Use underscores instead of spaces in column names

## plot

In [ ]:
df.plot(
    kind='scatter',
    x='age', 
    y='salary',
    title='age vs salary'
)

## vector operations 

In [ ]:
salary_limit = df['salary'].quantile(0.75)
df['result'] = df['salary'] > salary_limit
print(df.result.value_counts())
df['birthday_square'] = df['birthday'].dt.day ** 2
# df['x'] = df['y'].map({"YES": True, "NO": False})
df.head()

## Don't use inplace
Avoid using `inplace=True` as it is deprecated and will be removed in the future

e.g. prefer using `df = df.drop(...)` instead of `df.drop(..., inplace=True)`

In [ ]:
df = df.drop(columns=['result', 'birthday_square'])
df.head()

## Query Method
use `df.query(...)` instead of:
- `df[df['column'] == ...]` 
- or `df.loc[(...) & (...)]`

use the `@` symbol to access external variables in queries
- e.g. `@var_name`

In [ ]:
age_limit = 40
salary_limit = df['salary'].quantile(0.95)
print('age_limit:   ', age_limit)
print('salary_limit:', salary_limit.round(2))
print('')

# bad example
high_paid = df.loc[(df['age'] > age_limit) & (df['salary'] > salary_limit)]
print(high_paid[['name', 'age', 'salary']])
print('')

# better example
high_paid = df.query('age > @age_limit and salary > @salary_limit')
print(high_paid[['name', 'age', 'salary']])


## .copy() and .str
.str can be used to access string methods on a column

In [ ]:
# be sure to .copy() slices if operations are to be performed
df_young = df.query('age < @age_limit').copy()
# any string functions can be vectorized using df.str
df_young['first_name'] = df_young['name'].str.split(' ').str[0]
df_young.head()

## chaining
use chaining to avoid creating temporary variables

In [ ]:
age_limit = 35
print('max age:', df['age'].max())
print('min age:', df['age'].min())

# Categorize into age_groups, then calculate the average salary for each group
avg_salary = (
    df.assign(
        age_group=(df['age'] * 2 / 10) # every 5 years
            .astype('int32') # floor
            * 5
    )
    .groupby('age_group')
    ['salary']
    .agg(['mean', 'median', 'count', 'std', 'min', 'max'])
    .round(2)
    .assign(
        perc_change=lambda x: x['mean'].pct_change(),
        diff=lambda x: x['mean'].diff()
    )
)
# avg_salary['perc_change'] = avg_salary['mean'].pct_change()
# avg_salary['diff'] = avg_salary['mean'].diff()
"""
alternatives:

age_group=lambda x: np.where(x['age'] < age_limit, 'young', 'old')

age_group=(df['age'] < age_limit)
   .map({True: 'young', False: 'old'})

.agg(['mean', 'count', 'std', 'min', 'max', 'median', 'sum', 'sem', 'var', 'skew'])
"""
avg_salary


In [ ]:
def get_preferred_name(row):
    age_limit = 35  # Example age limit
    if row['age'] < age_limit:
        return row['name'].split(' ')[0]
    else:
        return row['name'].split(' ')[1]

# Vectorized operation to create 'preferred_name' based on 'age'
df['preferred_name'] = df.apply(
    get_preferred_name,
    axis=1 # axis=1 = apply once per row (use axis=0 for once per column)
)

# Display the first 10 rows of the 'preferred_name' column
print(df['preferred_name'].head(10))

## styling

In [ ]:
# gradiant styling example
(
    df.sort_values('salary')[['name', 'salary']]
    .reset_index(drop=True)
    .style
    .background_gradient(cmap='viridis', subset=['salary'])
)